## Verificación del fix de celdas fusionadas

Objetivo: comprobar, con el archivo real, que la versión corregida de
extraer_xlsx resuelve el problema detectado en 01_exploration.ipynb
(la cabecera de grupo "SERVICIO DE ORIENTACIÓN..." se perdía porque
openpyxl no propaga valores fuera de la celda superior-izquierda de
un rango fusionado).

Fuente del fix: commit 19b69c9, rama prueba/excel-celdas-fusionadas
(autor: isrodam).

In [ ]:
# Versión rota
from pathlib import Path
from openpyxl import load_workbook

DATA_DIR = Path.cwd().parent / "data" / "raw"
excel_path = DATA_DIR / "financiero_2025_indicadores_control_financiero.xlsx"

def extraer_xlsx_roto(ruta):
    wb = load_workbook(ruta, read_only=True, data_only=True)
    ws = wb["Empleo"]
    filas = [list(fila) for fila in ws.iter_rows(values_only=True)]
    return filas

filas_rotas = extraer_xlsx_roto(excel_path)
print("Fila de cabecera de grupo (ANTES del fix):")
print(filas_rotas[1])

## Como esperábamos: esta fila sale con `None` en casi todas las posiciones salvo donde empieza cada rango fusionado. Ahora la versión corregida.

In [ ]:
# Versión corregida
def _propagar_celdas_fusionadas(ws, filas: list) -> list:
    """Copia el valor de cada celda fusionada a TODAS las celdas que ocupa
    visualmente. openpyxl solo guarda el valor en la celda superior-
    izquierda del rango fusionado."""
    for rango in ws.merged_cells.ranges:
        valor = ws.cell(row=rango.min_row, column=rango.min_col).value
        for r in range(rango.min_row, rango.max_row + 1):
            idx_fila = r - ws.min_row
            if not (0 <= idx_fila < len(filas)):
                continue
            for c in range(rango.min_col, rango.max_col + 1):
                idx_col = c - ws.min_column
                if 0 <= idx_col < len(filas[idx_fila]):
                    filas[idx_fila][idx_col] = valor
    return filas


def extraer_xlsx_corregido(ruta, nombre_hoja):
    wb = load_workbook(ruta, data_only=True)  # sin read_only: necesitamos merged_cells
    ws = wb[nombre_hoja]
    filas = [list(fila) for fila in ws.iter_rows(values_only=True)]
    filas = _propagar_celdas_fusionadas(ws, filas)
    return filas

filas_corregidas = extraer_xlsx_corregido(excel_path, "Empleo")
print("Fila de cabecera de grupo (DESPUÉS del fix):")
print(filas_corregidas[1])

### Validación

Si el fix funciona, esta fila ya no debería tener `None` sueltos entre
"INDICADORES INFORME MENSUAL SEPE" y "SERVICIO DE ORIENTACIÓN..." —
cada columna del grupo debería llevar el valor de su título propagado.

In [ ]:
filas_sin_fusion = extraer_xlsx_corregido(excel_path, "Centro Formación")
print("Primeras 2 filas de una hoja SIN celdas fusionadas:")
print(filas_sin_fusion[0])
print(filas_sin_fusion[1])

### Validación de no-regresión

Con 0 celdas fusionadas, `ws.merged_cells.ranges` está vacío, así que el
bucle de `_propagar_celdas_fusionadas` no itera nada y `filas` vuelve
exactamente igual a como entró. Si esta salida se ve igual que antes del
fix (sin ningún None inesperado ni fila desplazada), confirmamos que el
fix es seguro también para hojas simples.

In [ ]:
from src_agents.rag.extractor_generico import extraer_xlsx
bloques = extraer_xlsx(excel_path)
print(bloques[0].etiqueta, "->", bloques[0].contenido[1])